In [0]:
from pyspark.sql.functions import col, sum, when, count, round

# 1. Leemos la capa Bronze en modo estático (Batch) para analizarla
bronze_table_path = "/Volumes/workspace/default/e_commerce/bronze/clickstream"
df_bronze_eda = spark.read.format("delta").load(bronze_table_path)

# 2. Obtenemos el total de registros (N)
total_rows = df_bronze_eda.count()
print(f"Total de registros en Bronze: {total_rows:,}")

# 3. Calculamos la proporción matemática de nulos por columna
# Creamos una lista de expresiones de agregación para Spark
exprs = [
    round((sum(when(col(c).isNull(), 1).otherwise(0)) / total_rows) * 100, 2).alias(f"{c}_null_pct")
    for c in df_bronze_eda.columns
]

# 4. Ejecutamos el cálculo distribuido
df_missing = df_bronze_eda.select(*exprs)

# Mostramos los resultados (Si usas Databricks, display() genera una tabla interactiva)
display(df_missing)